# Computing first baselines

## 1. Load Data & Define Feature Types

We load the un-encoded training set saved in Notebook 1. We then separate our target (`Churn`) and group numerical vs. categorical features for the Scikit-Learn Pipeline.

In [3]:
import pandas as pd
import numpy as np

# Load training data
train_df = pd.read_csv('../data/train_set.csv')

# Separate features and target
X = train_df.drop(columns=['Churn'])
y = (train_df['Churn'] == 'Yes').astype(int)

# Identify feature types
num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = X.select_dtypes(include=['object']).columns.tolist()

print(f"Numerical columns ({len(num_cols)}): {num_cols}")
print(f"Categorical columns ({len(cat_cols)}): {cat_cols}")

Numerical columns (4): ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']
Categorical columns (15): ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']


## 2. Define Trivial Baselines to Beat

Before evaluating a machine learning model, we must establish simple baseline benchmarks:
1. **Majority Class Predictor**: Predicts that no customer churns (`0`).
2. **Single Feature Heuristic (`Contract`)**: Predicts churn (`1`) if `Contract == 'Month-to-month'`, and no churn (`0`) otherwise.

In [7]:
from sklearn.metrics import roc_auc_score, accuracy_score

# Baseline 1: Predict majority class (all 0s)
y_pred_majority = np.zeros(len(y))

# Baseline 2: Single feature rule (Contract == 'Month-to-month')
y_pred_contract = (X['Contract'] == 'Month-to-month').astype(int)

print("--- Trivial Baseline Performance ---")
print(f"Majority Predictor Accuracy: {accuracy_score(y, y_pred_majority):.4f}")
print(f"Contract Heuristic Accuracy: {accuracy_score(y, y_pred_contract):.4f}")
print(f"Contract Heuristic ROC-AUC : {roc_auc_score(y, y_pred_contract):.4f}")

--- Trivial Baseline Performance ---
Majority Predictor Accuracy: 0.7346
Contract Heuristic Accuracy: 0.6548
Contract Heuristic ROC-AUC : 0.7289


## 3. Full Pipeline & Stratified K-Fold Cross-Validation

We construct a complete Scikit-Learn Pipeline using `ColumnTransformer`:
- **Numerical features**: `SimpleImputer(strategy='median')` -> `StandardScaler()`
- **Categorical features**: `SimpleImputer(strategy='most_frequent')` -> `OneHotEncoder(drop='first', handle_unknown='ignore')`
- **Model**: `LogisticRegression()`

The Pipeline is evaluated with **5-fold StratifiedKFold** to handle the ~26% churn class imbalance without data leakage.

In [5]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score

# 1. Preprocessing Transformers
num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'))
])

# 2. Combine into ColumnTransformer
preprocessor = ColumnTransformer(transformers=[
    ('num', num_transformer, num_cols),
    ('cat', cat_transformer, cat_cols)
])

# 3. Create full Logistic Regression Pipeline
full_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])

# 4. Stratified 5-Fold Cross Validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(full_pipeline, X, y, cv=cv, scoring='roc_auc')

# 5. Report Mean and Standard Deviation
print("--- Logistic Regression Baseline Pipeline ---")
print(f"ROC-AUC Scores across folds: {np.round(cv_scores, 4)}")
print(f"Mean ROC-AUC: {cv_scores.mean():.4f} +/- {cv_scores.std():.4f}")

--- Logistic Regression Baseline Pipeline ---
ROC-AUC Scores across folds: [0.8478 0.8252 0.842  0.8629 0.853 ]
Mean ROC-AUC: 0.8462 +/- 0.0126


## 4. Baseline Results & Synthesis

### Summary Comparison Table

| Model / Strategy | Type | Mean ROC-AUC | Std Dev ($\sigma$) | Status |
| :--- | :--- | :---: | :---: | :---: |
| **Majority Class Predictor** | Trivial Baseline | `0.5000` | `0.0000` | Benchmark |
| **Single Feature (`Contract`)** | Heuristic Baseline | `0.7289` | N/A | Benchmark |
| **Logistic Regression Pipeline** | Machine Learning | **`0.8462`** | **`0.0126`** | **To Beat** |

---

### Key Observations

1. **Significant Improvement Over Trivial Baselines:**
   - The **Majority Class Predictor** yields an ROC-AUC of `0.5000` (no discrimination ability).
   - The **Single-Feature Heuristic** (`Contract == 'Month-to-month'`) achieves an ROC-AUC of `~0.7289`, confirming that contract type is a strong individual predictor.
   - The **Logistic Regression Pipeline** achieves a mean ROC-AUC of **`0.8462`**, beating the best trivial baseline by **+0.7273 (+13.9%)**.

2. **Model Stability across Folds:**
   - The standard deviation across the 5 stratified folds is very low (**`0.0126`**), ranging from `0.8252` to `0.8629`.
   - This tight spread indicates high stability and confirms that the model is not overfitting to specific data splits.

3. **Leakage & Sanity Check:**
   - An ROC-AUC score of `0.8462` is strong but realistic for the Telco dataset.
   - The absence of near-1.0 scores confirms that no data leakage occurred during processing, as all transformations (imputation, scaling, one-hot encoding) were fit strictly within each cross-validation fold.